In [ ]:
import os
import sys

sys.path.append(os.path.abspath('..'))
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2' # 0=all messages, 1=information, 2=warning, 3=error

env_path = sys.prefix
os.environ['CC'] = f"{env_path}/bin/x86_64-conda-linux-gnu-cc"
os.environ['CXX'] = f"{env_path}/bin/x86_64-conda-linux-gnu-c++"

os.environ['PKG_CONFIG_PATH'] = f"{env_path}/lib/pkgconfig"
os.environ['LD_LIBRARY_PATH'] = f"{env_path}/lib"


import numpy as np
from fenics import *
from src.physics import constants as c
from src.visualization.pdeplots import N_GRID

os.makedirs('data/fem', exist_ok=True)


set_log_level(40)

In [ ]:
# MESH
nx = 400
ny = 400
mesh = UnitSquareMesh(nx,ny) # create a mesh in the unit square
V = FunctionSpace(mesh, 'CG', 1)

# BOUNDARY CONDITIONS
b_markers = MeshFunction('size_t', mesh, mesh.topology().dim() - 1)
b_markers.set_all(0) 


r = c.R_A_VAL
for f in facets(mesh):
    if f.exterior(): 
        x = f.midpoint().x()
        y = f.midpoint().y()
        dist = np.sqrt(x**2 + y**2)
        if np.isclose(dist, 0.0, atol=r):
            b_markers[f] = 1

psi_activation = Constant(0.0)
bc = DirichletBC(V, psi_activation, b_markers, 1)

v_teo = 62.0 * np.sqrt(1.529) 
u_initial = Expression(f'sqrt(pow(x[0] - x0, 2) + pow(x[1] - y0, 2)) / {v_teo}', degree=1, x0=0.0, y0=0.0)

N_points = N_GRID
x_test = np.linspace(0.0, 1.0, N_points)
y_test = np.linspace(0.0, 1.0, N_points)
X_test, Y_test = np.meshgrid(x_test, y_test)

x_flat = X_test.flatten()
y_flat = Y_test.flatten()
X_test_flattened = np.hstack((x_flat[:, None], y_flat[:, None]))
coords_fem = V.tabulate_dof_coordinates()

In [4]:
# margini + numero simulazioni
min_c = 0.2   
max_c = 0.8   
N_sim = 1000

cx_vals = np.random.uniform(min_c, max_c, N_sim)
cy_vals = np.random.uniform(min_c, max_c, N_sim)

# rimozione duplicati e shuffle
centers = list(set([(round(cx, 3), round(cy, 3)) for cx, cy in zip(cx_vals, cy_vals)]))
N_sim = len(centers)
np.random.shuffle(centers)

n_train = int(0.8 * N_sim)
n_val = int(0.1 * N_sim)

centers_train = centers[:n_train]
centers_val = centers[n_train:n_train+n_val]
centers_test = centers[n_train+n_val:]

np.savez('data/fem/centers_split.npz', train=centers_train, val=centers_val, test=centers_test)

print(f"Generati {len(centers)} centri per FEM")

Generati 998 centri per FEM


In [ ]:
# Data structures
X_train_list, U_train_list = [], []
X_val_list, U_val_list = [], []
X_test_list, U_test_list = [], []

X_grid_train_list, U_grid_train_list = [], []
X_grid_val_list, U_grid_val_list = [], []
X_grid_test_list, U_grid_test_list = [], []

n_punti_da_salvare = 1020 

M_tensor = Expression(
    '(x[0] >= cx - 0.1 && x[0] <= cx + 0.1 && x[1] >= cy - 0.1 && x[1] <= cy + 0.1) ? m_scar : m_normal',
    degree=0,
    cx=0.5, cy=0.5,
    m_scar=c.M_SCAR, m_normal=c.M_VAL
)

psi = Function(V)
v = TestFunction(V)
eps = Constant(c.EPS_VAL)
c0 = Constant(c.C0_VAL)

grad_psi = grad(psi)
M_grad_psi = M_tensor * grad_psi
norm_term = sqrt(inner(grad_psi, M_grad_psi))
F = (c0 * norm_term * v * dx + eps * inner(M_grad_psi, grad(v)) * dx - Constant(1.0) * v * dx)
J = derivative(F, psi)

pb = NonlinearVariationalProblem(F, psi, bc, J)
solver = NonlinearVariationalSolver(pb)

prm = solver.parameters
prm['newton_solver']['linear_solver'] = 'gmres'
prm['newton_solver']['preconditioner'] = 'amg'
prm['newton_solver']['absolute_tolerance'] = 1E-8
prm['newton_solver']['relative_tolerance'] = 1E-7
prm['newton_solver']['maximum_iterations'] = 50
prm['newton_solver']['error_on_nonconvergence'] = False

for i, (cx, cy) in enumerate(centers):

    M_tensor.cx = cx
    M_tensor.cy = cy

    
    print(f"progresso: {i}/{len(centers)} | centro: {cx}, {cy}")

    
    psi.interpolate(u_initial)
    
    n_it, converged = solver.solve()
    if not converged:
        print(f"Soluzione con centro {cx}, {cy} did not converge in {n_it} iterations.")

    # dati pinn 
    values_fem_all = psi.vector().get_local().reshape(-1, 1)
    x_mesh_all = coords_fem[:, 0]
    y_mesh_all = coords_fem[:, 1]
    
    nodi_totali = len(values_fem_all)
    indici_casuali = np.random.choice(nodi_totali, n_punti_da_salvare, replace=False)
    
    values_fem = values_fem_all[indici_casuali]
    x_mesh = x_mesh_all[indici_casuali]
    y_mesh = y_mesh_all[indici_casuali]
    
    cx_array = np.full_like(x_mesh, cx)
    cy_array = np.full_like(y_mesh, cy) 
    
    inputs_pinn = np.column_stack((x_mesh, y_mesh, cx_array, cy_array))
    outputs_pinn = values_fem

    # grid 
    u_fem_grid = np.array([psi(pt) for pt in X_test_flattened]).reshape(-1, 1)
    x_grid = X_test_flattened[:, 0]
    y_grid = X_test_flattened[:, 1]
    
    cx_grid = np.full_like(x_grid, cx)
    cy_grid = np.full_like(y_grid, cy)
    
    inputs_grid = np.column_stack((x_grid, y_grid, cx_grid, cy_grid))
    outputs_grid = u_fem_grid

    current_center = (cx, cy)
    
    if current_center in centers_train:
        X_train_list.append(inputs_pinn)
        U_train_list.append(outputs_pinn)
        X_grid_train_list.append(inputs_grid)  
        U_grid_train_list.append(outputs_grid)
        
    elif current_center in centers_val:
        X_val_list.append(inputs_pinn)
        U_val_list.append(outputs_pinn)
        X_grid_val_list.append(inputs_grid)
        U_grid_val_list.append(outputs_grid)

    else:
        X_test_list.append(inputs_pinn)
        U_test_list.append(outputs_pinn)
        X_grid_test_list.append(inputs_grid)
        U_grid_test_list.append(outputs_grid)

X_train = np.vstack(X_train_list)
U_train = np.vstack(U_train_list)
X_val = np.vstack(X_val_list)
U_val = np.vstack(U_val_list)
X_test = np.vstack(X_test_list)
U_test = np.vstack(U_test_list)

X_grid_train = np.vstack(X_grid_train_list)
U_grid_train = np.vstack(U_grid_train_list)
X_grid_val = np.vstack(X_grid_val_list)
U_grid_val = np.vstack(U_grid_val_list)
X_grid_test = np.vstack(X_grid_test_list)
U_grid_test = np.vstack(U_grid_test_list)

# dati per pinn
np.savez('data/fem/fem_parametric_train.npz', coords=X_train, values=U_train)
np.savez('data/fem/fem_parametric_val.npz', coords=X_val, values=U_val)
np.savez('data/fem/fem_parametric_test.npz', coords=X_test, values=U_test)

# dati griglia èer vae
np.savez('data/fem/fem_parametric_train_grid.npz', coords=X_grid_train, values=U_grid_train)
np.savez('data/fem/fem_parametric_val_grid.npz', coords=X_grid_val, values=U_grid_val)
np.savez('data/fem/fem_parametric_test_grid.npz', coords=X_grid_test, values=U_grid_test)

print(f"Shape Train: {X_train.shape} | Val: {X_val.shape} | Test: {X_test.shape}")
print(f"Shape Train Griglia: {X_grid_train.shape} | Val Griglia: {X_grid_val.shape} | Test Griglia: {X_grid_test.shape}")


progresso: 0/998 | centro: 0.607, 0.787
progresso: 1/998 | centro: 0.298, 0.306
progresso: 2/998 | centro: 0.21, 0.604
progresso: 3/998 | centro: 0.277, 0.645
progresso: 4/998 | centro: 0.485, 0.646
progresso: 5/998 | centro: 0.23, 0.371
progresso: 6/998 | centro: 0.777, 0.395
progresso: 7/998 | centro: 0.545, 0.399
progresso: 8/998 | centro: 0.249, 0.259
progresso: 9/998 | centro: 0.38, 0.293
progresso: 10/998 | centro: 0.342, 0.501
progresso: 11/998 | centro: 0.747, 0.414
progresso: 12/998 | centro: 0.256, 0.709
progresso: 13/998 | centro: 0.715, 0.508
progresso: 14/998 | centro: 0.318, 0.763
progresso: 15/998 | centro: 0.763, 0.526
progresso: 16/998 | centro: 0.352, 0.354
progresso: 17/998 | centro: 0.314, 0.485
progresso: 18/998 | centro: 0.303, 0.494
progresso: 19/998 | centro: 0.73, 0.505
progresso: 20/998 | centro: 0.634, 0.38
progresso: 21/998 | centro: 0.261, 0.774
progresso: 22/998 | centro: 0.523, 0.234
progresso: 23/998 | centro: 0.756, 0.239
progresso: 24/998 | centro: 0.6